In [1]:
# Import the libarys 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
df = pd.read_csv("/kaggle/input/datasets/mahmoudahmed6/udemy-top-5k-course-2022/data.csv")

In [3]:
df.head(2)

,course_name,instructor,course url,course image,course description,reviews_avg,reviews_count,course_duration,lectures_count,level,price_after_discount,main_price,course_flag,students_count,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,2022 Complete Python Bootcamp From Zero to Her...,Jose Portilla,https://www.udemy.com/course/complete-python-b...,https://img-b.udemycdn.com/course/240x135/5678...,Learn Python like a Professional Start from t...,Rating: 4.6 out of 5,440383 reviews,22 total hours,155 lectures,All Levels,Current price: E£319.99,"Original price: E£1,399.99",NaN,"1,629,692 students",NaN,NaN,NaN,NaN
1,The Web Developer Bootcamp 2022,Colt Steele,https://www.udemy.com/course/the-web-developer...,https://img-b.udemycdn.com/course/240x135/6252...,COMPLETELY REDONE - The only course you need t...,Rating: 4.7 out of 5,248508 reviews,64 total hours,615 lectures,All Levels,Current price: E£269.99,"Original price: E£1,399.99",NaN,"830,559 students",NaN,NaN,NaN,NaN


In [4]:
df.shape

(5027, 18)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5027 entries, 0 to 5026
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   course_name           5013 non-null   object
 1   instructor            5016 non-null   object
 2   course url            5020 non-null   object
 3   course image          5017 non-null   object
 4   course description    5016 non-null   object
 5   reviews_avg           5017 non-null   object
 6   reviews_count         5017 non-null   object
 7   course_duration       5017 non-null   object
 8   lectures_count        5017 non-null   object
 9   level                 5017 non-null   object
 10  price_after_discount  5017 non-null   object
 11  main_price            4799 non-null   object
 12  course_flag           518 non-null    object
 13  students_count        5013 non-null   object
 14  Unnamed: 14           3 non-null      object
 15  Unnamed: 15           1 non-null      

In [6]:
df.isnull().sum()

course_name               14
instructor                11
course url                 7
course image              10
course description        11
reviews_avg               10
reviews_count             10
course_duration           10
lectures_count            10
level                     10
price_after_discount      10
main_price               228
course_flag             4509
students_count            14
Unnamed: 14             5024
Unnamed: 15             5026
Unnamed: 16             5026
Unnamed: 17             5026
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(22)

In [ ]:
df.drop_duplicates(keep="last", inplace=True)
df.reset_index(drop=True, inplace=True)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.shape


(5005, 18)

# Drop unused columns

In [11]:
def drop_columns(df, columns_list):
    df = df.drop(columns=columns_list)
    return df

columns_list = ["Unnamed: 14", "Unnamed: 15", "Unnamed: 16", "Unnamed: 17", "course_flag"]
df = drop_columns(df, columns_list)

# Data cleaning on numerical columns

In [12]:
def clean_numeric_column(df, column_name, remove_commas=False):
    col = df[column_name]
    if remove_commas:
        col = col.str.replace(",", "", regex=False)
    df[column_name] = col.str.extract(r"(\d+\.?\d*)").astype(float)
    return df

df = clean_numeric_column(df, "reviews_avg")
df = clean_numeric_column(df, "reviews_count", remove_commas=True)
df = clean_numeric_column(df, "course_duration")
df = clean_numeric_column(df, "lectures_count")
df = clean_numeric_column(df, "price_after_discount", remove_commas=True)
df = clean_numeric_column(df, "main_price",remove_commas=True)
df = clean_numeric_column(df, "students_count",remove_commas=True)


# Fill the missing values

In [13]:
def fill_categorical(df, column_name):
    mode_value = df[column_name].mode()[0]
    df[column_name] = df[column_name].fillna(mode_value)
    return df

df = fill_categorical(df, "course_name")
df = fill_categorical(df, "instructor")

In [14]:
def unfill(df, column_name):
    df = df.dropna(subset=[column_name])
    return df

df = unfill(df, "course url")
df = unfill(df, "course image")
df = unfill(df, "course description")

In [ ]:
def fill_numerical(data, column_name):
    median_value = data[column_name].median()
    data[column_name] = data[column_name].fillna(median_value)
    return data

df = fill_numerical(df, "reviews_avg")
df = fill_numerical(df, "reviews_count")
df = fill_numerical(df, "lectures_count")
df = fill_numerical(df, "price_after_discount")
df = fill_numerical(df, "main_price")
df = fill_numerical(df, "students_count")

In [16]:
df.isnull().sum()

course_name             0
instructor              0
course url              0
course image            0
course description      0
reviews_avg             0
reviews_count           0
course_duration         0
lectures_count          0
level                   0
price_after_discount    0
main_price              0
students_count          0
dtype: int64

In [17]:
df.sample()

,course_name,instructor,course url,course image,course description,reviews_avg,reviews_count,course_duration,lectures_count,level,price_after_discount,main_price,students_count
3845,Android P with Machine Learning apps - 26+ pro...,Igneus Technologies,https://www.udemy.com/course/android-p-with-ma...,https://img-b.udemycdn.com/course/240x135/1950...,Master Android development with latest tech st...,4.7,199.0,19.5,127.0,All Levels,229.99,1199.99,912.0


# Feature Engineering 

In [ ]:
df["course_text"] = (
    "course: " + df["course_name"].fillna("").astype(str) + " " +
    "description: " + df["course description"].fillna("").astype(str) + " " +
    "instructor: " + df["instructor"].fillna("").astype(str) + " " +
    "level: " + df["level"].fillna("").astype(str)
)

In [19]:
df["log_students"] = np.log1p(
    df["students_count"].fillna(0)
)

df["log_reviews"] = np.log1p(
    df["reviews_count"].fillna(0)
)

In [20]:
def min_max_scale(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            0.0,
            index=series.index
        )

    return (series - min_value) / (max_value - min_value)

In [21]:
df["students_score"] = min_max_scale(
    df["log_students"]
)

df["reviews_score"] = min_max_scale(
    df["log_reviews"]
)

df["rating_score"] = min_max_scale(
    df["reviews_avg"].fillna(
        df["reviews_avg"].median()
    )
)

In [22]:
df["popularity_score"] = (
    0.40 * df["students_score"] +
    0.25 * df["reviews_score"] +
    0.25 * df["rating_score"] 
)

In [23]:
df[
    [
        "course_name",
        "students_count",
        "reviews_count",
        "reviews_avg",
        "popularity_score"
    ]
].sort_values(
    "popularity_score",
    ascending=False
).head(5)

,course_name,students_count,reviews_count,reviews_avg,popularity_score
0,2022 Complete Python Bootcamp From Zero to Her...,1629692.0,440383.0,4.6,0.651820
1,The Web Developer Bootcamp 2022,830559.0,248508.0,4.7,0.608427
2,The Complete 2022 Web Development Bootcamp,794897.0,234837.0,4.7,0.605177
6,Machine Learning A-Z™: Hands-On Python & R In ...,900823.0,163104.0,4.5,0.602856
11,Automate the Boring Stuff with Python Programming,1060445.0,103366.0,4.7,0.600539


In [24]:
df = df.drop(
    columns=[
        "log_students",
        "log_reviews"
    ],
    errors="ignore"
)

In [25]:
df.head(2)

,course_name,instructor,course url,course image,course description,reviews_avg,reviews_count,course_duration,lectures_count,level,price_after_discount,main_price,students_count,course_text,students_score,reviews_score,rating_score,popularity_score
0,2022 Complete Python Bootcamp From Zero to Her...,Jose Portilla,https://www.udemy.com/course/complete-python-b...,https://img-b.udemycdn.com/course/240x135/5678...,Learn Python like a Professional Start from t...,4.6,440383.0,22.0,155.0,All Levels,319.99,1399.99,1629692.0,course: 2022 Complete Python Bootcamp From Zer...,1.00000,1.000000,0.007281,0.651820
1,The Web Developer Bootcamp 2022,Colt Steele,https://www.udemy.com/course/the-web-developer...,https://img-b.udemycdn.com/course/240x135/6252...,COMPLETELY REDONE - The only course you need t...,4.7,248508.0,64.0,615.0,All Levels,269.99,1399.99,830559.0,course: The Web Developer Bootcamp 2022 descri...,0.92254,0.950113,0.007532,0.608427


In [26]:
df.columns

Index(['course_name', 'instructor', 'course url', 'course image',
       'course description', 'reviews_avg', 'reviews_count', 'course_duration',
       'lectures_count', 'level', 'price_after_discount', 'main_price',
       'students_count', 'course_text', 'students_score', 'reviews_score',
       'rating_score', 'popularity_score'],
      dtype='object')

In [27]:
popular_courses = (
    df.sort_values(
        "popularity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

In [28]:
df.sort_values(
    "popularity_score",
    ascending=False
).reset_index(drop=True).head(5)

,course_name,instructor,course url,course image,course description,reviews_avg,reviews_count,course_duration,lectures_count,level,price_after_discount,main_price,students_count,course_text,students_score,reviews_score,rating_score,popularity_score
0,2022 Complete Python Bootcamp From Zero to Her...,Jose Portilla,https://www.udemy.com/course/complete-python-b...,https://img-b.udemycdn.com/course/240x135/5678...,Learn Python like a Professional Start from t...,4.6,440383.0,22.0,155.0,All Levels,319.99,1399.99,1629692.0,course: 2022 Complete Python Bootcamp From Zer...,1.000000,1.000000,0.007281,0.651820
1,The Web Developer Bootcamp 2022,Colt Steele,https://www.udemy.com/course/the-web-developer...,https://img-b.udemycdn.com/course/240x135/6252...,COMPLETELY REDONE - The only course you need t...,4.7,248508.0,64.0,615.0,All Levels,269.99,1399.99,830559.0,course: The Web Developer Bootcamp 2022 descri...,0.922540,0.950113,0.007532,0.608427
2,The Complete 2022 Web Development Bootcamp,Dr. Angela Yu,https://www.udemy.com/course/the-complete-web-...,https://img-b.udemycdn.com/course/240x135/1565...,Become a Full-Stack Web Developer with just ON...,4.7,234837.0,65.5,490.0,All Levels,349.99,1699.99,794897.0,course: The Complete 2022 Web Development Boot...,0.917496,0.945180,0.007532,0.605177
3,Machine Learning A-Z™: Hands-On Python & R In ...,"Kirill Eremenko, Hadelin de Ponteves, Ligency ...",https://www.udemy.com/course/machinelearning/,https://img-b.udemycdn.com/course/240x135/9503...,Learn to create Machine Learning Algorithms in...,4.5,163104.0,42.5,287.0,All Levels,269.99,1299.99,900823.0,course: Machine Learning A-Z™: Hands-On Python...,0.931872,0.913399,0.007030,0.602856
4,Automate the Boring Stuff with Python Programming,Al Sweigart,https://www.udemy.com/course/automate/,https://img-b.udemycdn.com/course/240x135/5436...,A practical programming course for office work...,4.7,103366.0,9.5,51.0,All Levels,319.99,479.99,1060445.0,course: Automate the Boring Stuff with Python ...,0.950619,0.873632,0.007532,0.600539


In [29]:
df["course_id"] = [
    str(i).zfill(4)
    for i in range(1, len(df) + 1)
]

# Popularity recommendation¶

In [30]:
def popular_recommendations(n=10,level=None):
    data = df.copy()

    if level is not  None:
        data = data[
        data["level"].str.lower() == level.lower()
        ]
    recommendations = (
        data.sort_values(
             "popularity_score",
             ascending = False
        ).head(n)
    )

    return recommendations[
    [
            "course_id",
            "course_name",
            "level",
            "reviews_avg",
            "reviews_count",
            "students_count",
            "popularity_score"
        ]
    ]

In [31]:
popular_recommendations(5)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,popularity_score
0,0001,2022 Complete Python Bootcamp From Zero to Her...,All Levels,4.6,440383.0,1629692.0,0.651820
1,0002,The Web Developer Bootcamp 2022,All Levels,4.7,248508.0,830559.0,0.608427
2,0003,The Complete 2022 Web Development Bootcamp,All Levels,4.7,234837.0,794897.0,0.605177
6,0007,Machine Learning A-Z™: Hands-On Python & R In ...,All Levels,4.5,163104.0,900823.0,0.602856
11,0012,Automate the Boring Stuff with Python Programming,All Levels,4.7,103366.0,1060445.0,0.600539


In [32]:
popular_recommendations(
    n=10,
    level="All Levels"
)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,popularity_score
0,0001,2022 Complete Python Bootcamp From Zero to Her...,All Levels,4.6,440383.0,1629692.0,0.651820
1,0002,The Web Developer Bootcamp 2022,All Levels,4.7,248508.0,830559.0,0.608427
2,0003,The Complete 2022 Web Development Bootcamp,All Levels,4.7,234837.0,794897.0,0.605177
6,0007,Machine Learning A-Z™: Hands-On Python & R In ...,All Levels,4.5,163104.0,900823.0,0.602856
11,0012,Automate the Boring Stuff with Python Programming,All Levels,4.7,103366.0,1060445.0,0.600539
4,0005,Java Programming Masterclass covering Java 11 ...,All Levels,4.5,171838.0,727934.0,0.594198
7,0008,The Complete JavaScript Course 2023: From Zero...,All Levels,4.7,156031.0,699335.0,0.590377
5,0006,"React - The Complete Guide (incl Hooks, React ...",All Levels,4.6,166128.0,649985.0,0.588317
3,0004,Angular - The Complete Guide (2023 Edition),All Levels,4.6,174576.0,634196.0,0.588268
8,0009,100 Days of Code: The Complete Python Pro Boot...,All Levels,4.7,150024.0,662274.0,0.587019


In [33]:
def get_popular_course_ids(n=10):
    return(
        df.sort_values(
            "popularity_score",
            ascending=False
        ).head(n)["course_id"].tolist()
    )

#  TF-IDF Content-Based Recommender

In [34]:
# course_text
#       ↓
# TF-IDF
#       ↓
# Course vectors
#       ↓
# Cosine similarity
#       ↓
# Similar courses

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
tfidf = TfidfVectorizer(
    lowercase = True,
    stop_words = "english",
    max_features = 20000,
    ngram_range = (1,2),
    min_df = 2,
    max_df = 0.95,
    sublinear_tf=True
)

In [37]:
tfidf_matrix = tfidf.fit_transform(
    df["course_text"]
)

In [38]:
tfidf_matrix


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 150266 stored elements and shape (5000, 16505)>

In [39]:
print("Matrix shape:", tfidf_matrix.shape)

Matrix shape: (5000, 16505)


In [40]:
cosine_similarity(tfidf_matrix)

array([[1.        , 0.03103257, 0.03200101, ..., 0.01535283, 0.        ,
        0.01900485],
       [0.03103257, 1.        , 0.15818269, ..., 0.02251078, 0.00944771,
        0.        ],
       [0.03200101, 0.15818269, 1.        , ..., 0.        , 0.00872916,
        0.        ],
       ...,
       [0.01535283, 0.02251078, 0.        , ..., 1.        , 0.02315169,
        0.01133647],
       [0.        , 0.00944771, 0.00872916, ..., 0.02315169, 1.        ,
        0.05003862],
       [0.01900485, 0.        , 0.        , ..., 0.01133647, 0.05003862,
        1.        ]])

In [41]:
course_id_to_index = pd.Series(
    df.index,
    index = df["course_id"]
).to_dict()

In [42]:
def tfidf_recommend(
    course_id,
    n=10
):
    if course_id not in course_id_to_index:
        raise ValueError(
            f"Course ID {course_id} not found."
        )

    course_index = course_id_to_index[course_id]

    course_vector = tfidf_matrix[
        course_index
    ]

    similarity_scores = cosine_similarity(
        course_vector,
        tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(
        similarity_scores
    )[::-1]

    similar_indices = [
        idx
        for idx in similar_indices
        if idx != course_index
    ]

    top_indices = similar_indices[:n]

    recommendations = df.iloc[
        top_indices
    ].copy()

    recommendations["similarity_score"] = (
        similarity_scores[top_indices]
    )

    return recommendations[
        [
            "course_id",
            "course_name",
            "level",
            "reviews_avg",
            "reviews_count",
            "students_count",
            "similarity_score"
        ]
    ]

In [43]:
course_id = df.iloc[0]["course_id"]

print(
    df.loc[
        df["course_id"] == course_id,
        "course_name"
    ].iloc[0]
)

2022 Complete Python Bootcamp From Zero to Hero in Python


In [44]:
tfidf_recommend(
    course_id,
    n = 10
)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,similarity_score
2048,2044,Crash Course to Learn Python coding from basic...,Beginner,3.4,559.0,42490.0,0.592753
4826,4803,Complete Ruby Programmer - Master Ruby,All Levels,4.5,133.0,845.0,0.253984
2744,2739,Ansible Essentials with Hands-on Labs,All Levels,4.4,347.0,3418.0,0.232647
545,0545,Complete Python 3 Masterclass Journey,All Levels,3.9,3110.0,20601.0,0.216804
3221,3214,Flutter & Dart Bootcamp-Getting Started Comple...,All Levels,3.9,266.0,16884.0,0.213556
2933,2928,Complete Python Course: from Basics to Brillia...,All Levels,3.7,309.0,12771.0,0.209133
4184,4173,Python Programming Bootcamp 2022 from Basics t...,All Levels,4.2,174.0,18950.0,0.208098
4783,4761,Complete MLOps Bootcamp | From Zero to Hero in...,All Levels,4.3,135.0,1051.0,0.205393
191,0192,2022 Python for Machine Learning & Data Scienc...,All Levels,4.7,9855.0,76171.0,0.204165
4777,4755,Python for Absolute Beginners® | Zero to Exper...,All Levels,3.2,136.0,12709.0,0.202404


In [45]:
df.columns

Index(['course_name', 'instructor', 'course url', 'course image',
       'course description', 'reviews_avg', 'reviews_count', 'course_duration',
       'lectures_count', 'level', 'price_after_discount', 'main_price',
       'students_count', 'course_text', 'students_score', 'reviews_score',
       'rating_score', 'popularity_score', 'course_id'],
      dtype='object')

In [46]:
# test with specific course
course = df[
    df["course_name"].str.contains(
        "java",
        case=False,
        na=False
    )
]

course[
    [
        "course_id",
        "course_name"
    ]
].head(10)

,course_id,course_name
4,0005,Java Programming Masterclass covering Java 11 ...
7,0008,The Complete JavaScript Course 2023: From Zero...
14,0015,Selenium WebDriver with Java -Basics to Advanc...
32,0033,JavaScript: Understanding the Weird Parts
55,0056,Modern JavaScript From The Beginning
57,0058,Java Programming for Complete Beginners
67,0068,Rest API Testing (Automation) from Scratch-Res...
71,0072,JavaScript Basics for Beginners
77,0078,JavaScript - The Complete Guide 2023 (Beginner...
85,0086,JavaScript Algorithms and Data Structures Mast...


In [47]:
course.iloc[0]["course_id"]

'0005'

In [48]:
course_id = course.iloc[0]["course_id"]

tfidf_recommend( course_id, n=10)


,course_id,course_name,level,reviews_avg,reviews_count,students_count,similarity_score
520,0520,Learn Java Programming Crash Course,Beginner,4.6,3339.0,16329.0,0.404512
462,0463,Kotlin for Java Developers,All Levels,4.5,3694.0,21875.0,0.324106
4818,4795,The Complete C Course,Beginner,4.1,133.0,4168.0,0.314031
932,0932,Learn C# for Beginners Crash Course,Beginner,4.5,1639.0,8912.0,0.313068
3618,3608,Java Programming: Learn Core Java and Improve ...,All Levels,4.3,221.0,1288.0,0.310709
294,0295,Java Spring Tutorial Masterclass - Learn Sprin...,All Levels,4.5,6076.0,37445.0,0.300673
134,0135,Java Programming: Complete Beginner to Advanced,Beginner,4.2,13879.0,522455.0,0.292189
504,0504,Introduction to Continuous Integration & Conti...,All Levels,4.5,3415.0,7586.0,0.289962
13,0014,Learn Python Programming Masterclass,All Levels,4.6,94413.0,394722.0,0.286178
1433,1432,The Java Design Patterns Course,Intermediate,4.6,927.0,10578.0,0.262712


In [49]:
import joblib
from pathlib import Path

TFIDF_DIR = Path("/kaggle/working/tfidf")
TFIDF_DIR.mkdir(parents=True, exist_ok=True)   

# Save your model/vectorizer
joblib.dump(tfidf, TFIDF_DIR / "tfidf_vectorizer.pkl")

['/kaggle/working/tfidf/tfidf_vectorizer.pkl']

In [50]:
loaded_model = joblib.load(TFIDF_DIR / "tfidf_vectorizer.pkl")

In [51]:
# save matrix
from scipy.sparse import save_npz

save_npz(
    TFIDF_DIR / "tfidf_matrix.npz",
    tfidf_matrix
)

In [52]:
joblib.dump(
    course_id_to_index,
    TFIDF_DIR / "course_id_to_index.pkl"
)

['/kaggle/working/tfidf/course_id_to_index.pkl']

# Semantic Embedding Recommender

In [53]:
# Architecture:

#   Course description
#           ↓
#   Sentence Transformer
#           ↓
#   Embedding
#           ↓
#   FAISS
#           ↓
#   Nearest courses

In [54]:
from sentence_transformers import SentenceTransformer

In [55]:
df["course_text"] = df["course_text"].astype(str)

In [56]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# If you want to know more about this model, you can check this link: https://www.educative.io/answers/what-is-all-minilm-l6-v2-model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [57]:
embeddings = model.encode(
    df["course_text"].tolist(),
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings = True
)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [58]:
print(embeddings.shape)

(5000, 384)


In [59]:
# test semantic similarity
similarity_scores = (
    embeddings @ embeddings[0]
)

In [60]:
top_indices = np.argsort(
    similarity_scores
)[::-1][:12]

df.iloc[
    top_indices
][
    [
        "course_id",
        "course_name"
    ]
]

,course_id,course_name
0,0001,2022 Complete Python Bootcamp From Zero to Her...
2195,2191,Complete Python 3 Programming Bootcamp: Beginn...
4241,4230,Complete Python Bootcamp 2022: With Practical ...
4184,4173,Python Programming Bootcamp 2022 from Basics t...
4383,4372,Python Programming Complete Beginners Course B...
375,0376,Learn to Code with Python
1057,1056,Master Python Programming: The Complete Python...
8,0009,100 Days of Code: The Complete Python Pro Boot...
1603,1600,Python Programming Complete Beginners Course B...
4559,4544,Python Programming Bootcamp 2021 | Master Pyth...


In [ ]:
%pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [62]:
import faiss

In [63]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

In [64]:
index.add(
    embeddings.astype("float32")
)

In [65]:
course_id_to_index = pd.Series(
    df.index,
    index=df["course_id"]
).to_dict()

In [66]:
# recommender function
def semantic_recommend(
    course_id,
    n=10
):
    if course_id not in course_id_to_index:
        raise ValueError(
            f"Course ID {course_id} not found."
        )

    course_index = course_id_to_index[
        course_id
    ]

    query_vector = embeddings[
        course_index
    ].astype("float32").reshape(1, -1)

    scores, indices = index.search(
        query_vector,
        n + 1
    )

    result_indices = indices[0]
    result_scores = scores[0]

    results = []

    for idx, score in zip(
        result_indices,
        result_scores
    ):
        if idx == course_index:
            continue

        results.append(
            {
                "course_id": df.iloc[idx]["course_id"],
                "course_name": df.iloc[idx]["course_name"],
                "level": df.iloc[idx]["level"],
                "reviews_avg": df.iloc[idx]["reviews_avg"],
                "reviews_count": df.iloc[idx]["reviews_count"],
                "students_count": df.iloc[idx]["students_count"],
                "semantic_score": float(score)
            }
        )

        if len(results) == n:
            break

    return pd.DataFrame(results)

In [67]:
# test
course_id=df.iloc[0]["course_id"]
semantic_recommend(
    course_id,
    n = 10
)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,semantic_score
0,2191,Complete Python 3 Programming Bootcamp: Beginn...,All Levels,4.5,503.0,5106.0,0.871376
1,4230,Complete Python Bootcamp 2022: With Practical ...,All Levels,4.1,168.0,11197.0,0.829630
2,4173,Python Programming Bootcamp 2022 from Basics t...,All Levels,4.2,174.0,18950.0,0.822891
3,4372,Python Programming Complete Beginners Course B...,All Levels,4.8,158.0,9523.0,0.815200
4,0376,Learn to Code with Python,All Levels,4.7,4589.0,69274.0,0.808486
5,1056,Master Python Programming: The Complete Python...,All Levels,4.6,1393.0,10712.0,0.803323
6,0009,100 Days of Code: The Complete Python Pro Boot...,All Levels,4.7,150024.0,662274.0,0.800744
7,1600,Python Programming Complete Beginners Course B...,Beginner,4.6,786.0,57540.0,0.793355
8,4544,Python Programming Bootcamp 2021 | Master Pyth...,Beginner,4.3,148.0,20628.0,0.792989
9,1373,Python : Comprehensive Bootcamp (Beginner To ...,Beginner,4.3,984.0,47858.0,0.785762


In [68]:
# Save Embeddings
EMBEDDING_DIR = Path("/kaggle/working/embeddings")

EMBEDDING_DIR.mkdir(
    parents = True,
    exist_ok = True 
)

In [69]:
np.save(
    EMBEDDING_DIR / "course_embeddings.npy",
    embeddings
)

In [70]:
# Save the faiss index
faiss.write_index(
    index,
    str(
        EMBEDDING_DIR /
        "course_index.faiss"
    )
)

In [71]:
with open(
    EMBEDDING_DIR / "model_name.txt",
    "w"
) as f:
    f.write("all-MiniLM-L6-v2")

# Hybrid Recommender

In [72]:
# Semantic similarity
# +
# TF-IDF similarity
# +
# Popularity
# +
# Rating

In [73]:
from sklearn.preprocessing import MinMaxScaler

# load the models

In [74]:
from scipy.sparse import load_npz

tfidf = joblib.load(
    "/kaggle/working/tfidf/tfidf_vectorizer.pkl"
)

tfidf_matrix = load_npz(
    "/kaggle/working/tfidf/tfidf_matrix.npz"
)

course_id_to_index = joblib.load(
    "/kaggle/working/tfidf/course_id_to_index.pkl"
)

In [75]:
embeddings = np.load("/kaggle/working/embeddings/course_embeddings.npy")

In [76]:
import faiss

index = faiss.read_index(
    "/kaggle/working/embeddings/course_index.faiss"
)

In [77]:
from sklearn.metrics.pairwise import cosine_similarity

# Hybrid Fucation

In [78]:
def hybrid_recommend(
    course_id,
    n=10,
    semantic_weight=0.50,
    tfidf_weight=0.20,
    popularity_weight=0.20,
    rating_weight=0.10
):

    if course_id not in course_id_to_index:
        raise ValueError(
            f"Course ID {course_id} not found."
        )

    course_index = course_id_to_index[
        course_id
    ]

    # -----------------------------
    # 1. Semantic similarity
    # -----------------------------

    query_vector = embeddings[
        course_index
    ].astype("float32").reshape(1, -1)

    semantic_scores, semantic_indices = (
        index.search(
            query_vector,
            min(len(df), 100)
        )
    )

    candidate_indices = semantic_indices[0]

    semantic_score_map = {
        idx: float(score)
        for idx, score in zip(
            semantic_indices[0],
            semantic_scores[0]
        )
    }

    # -----------------------------
    # 2. TF-IDF similarity
    # -----------------------------

    course_vector = tfidf_matrix[
        course_index
    ]

    tfidf_scores = cosine_similarity(
        course_vector,
        tfidf_matrix[candidate_indices]
    ).flatten()

    # -----------------------------
    # 3. Build candidate dataframe
    # -----------------------------

    candidates = df.iloc[
        candidate_indices
    ].copy()

    candidates["semantic_score"] = [
        semantic_score_map[idx]
        for idx in candidate_indices
    ]

    candidates["tfidf_score"] = (
        tfidf_scores
    )

    # Remove selected course
    candidates = candidates[
        candidates["course_id"] != course_id
    ].copy()

    # -----------------------------
    # 4. Normalize popularity
    # -----------------------------

    candidates["popularity_norm"] = (
        candidates["popularity_score"]
    )

    # -----------------------------
    # 5. Normalize rating
    # -----------------------------

    rating_min = df["reviews_avg"].min()
    rating_max = df["reviews_avg"].max()

    if rating_max != rating_min:

        candidates["rating_norm"] = (
            candidates["reviews_avg"] -
            rating_min
        ) / (
            rating_max -
            rating_min
        )

    else:

        candidates["rating_norm"] = 0.0

    # -----------------------------
    # 6. Final score
    # -----------------------------

    candidates["hybrid_score"] = (
        semantic_weight *
        candidates["semantic_score"]
        +
        tfidf_weight *
        candidates["tfidf_score"]
        +
        popularity_weight *
        candidates["popularity_norm"]
        +
        rating_weight *
        candidates["rating_norm"]
    )

    # -----------------------------
    # 7. Sort
    # -----------------------------

    recommendations = (
        candidates
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(n)
    )

    return recommendations[
        [
            "course_id",
            "course_name",
            "level",
            "reviews_avg",
            "reviews_count",
            "students_count",
            "semantic_score",
            "tfidf_score",
            "popularity_norm",
            "rating_norm",
            "hybrid_score"
        ]
    ].reset_index(drop=True)

In [79]:
course_id = df.iloc[0]["course_id"]

hybrid_recommend(course_id, n = 10)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,semantic_score,tfidf_score,popularity_norm,rating_norm,hybrid_score
0,2044,Crash Course to Learn Python coding from basic...,Beginner,3.4,559.0,42490.0,0.733064,0.592753,0.338098,0.004268,0.553129
1,0009,100 Days of Code: The Complete Python Pro Boot...,All Levels,4.7,150024.0,662274.0,0.800744,0.083854,0.587019,0.007532,0.535300
2,0376,Learn to Code with Python,All Levels,4.7,4589.0,69274.0,0.808486,0.201887,0.407238,0.007532,0.526821
3,2191,Complete Python 3 Programming Bootcamp: Beginn...,All Levels,4.5,503.0,5106.0,0.871376,0.176791,0.239102,0.007030,0.519569
4,0290,Python Programming - From Basics to Advanced l...,All Levels,4.3,6229.0,319289.0,0.782081,0.152240,0.483884,0.006528,0.518918
5,0010,Python for Data Science and Machine Learning B...,All Levels,4.6,122487.0,588524.0,0.730083,0.172953,0.577109,0.007281,0.515782
6,4173,Python Programming Bootcamp 2022 from Basics t...,All Levels,4.2,174.0,18950.0,0.822891,0.208098,0.276131,0.006277,0.508919
7,4372,Python Programming Complete Beginners Course B...,All Levels,4.8,158.0,9523.0,0.815200,0.193699,0.242790,0.007783,0.495676
8,0545,Complete Python 3 Masterclass Journey,All Levels,3.9,3110.0,20601.0,0.766156,0.216804,0.342513,0.005524,0.495494
9,4230,Complete Python Bootcamp 2022: With Practical ...,All Levels,4.1,168.0,11197.0,0.829630,0.128690,0.251124,0.006026,0.491380


In [80]:
selected_course = df[
    df["course_id"] == course_id
]

selected_course[
    [
        "course_id",
        "course_name",
        "course description",
        "level"
    ]
]

,course_id,course_name,course description,level
0,0001,2022 Complete Python Bootcamp From Zero to Her...,Learn Python like a Professional Start from t...,All Levels


In [81]:
hybrid_recommend(course_id, n =5)

,course_id,course_name,level,reviews_avg,reviews_count,students_count,semantic_score,tfidf_score,popularity_norm,rating_norm,hybrid_score
0,2044,Crash Course to Learn Python coding from basic...,Beginner,3.4,559.0,42490.0,0.733064,0.592753,0.338098,0.004268,0.553129
1,0009,100 Days of Code: The Complete Python Pro Boot...,All Levels,4.7,150024.0,662274.0,0.800744,0.083854,0.587019,0.007532,0.535300
2,0376,Learn to Code with Python,All Levels,4.7,4589.0,69274.0,0.808486,0.201887,0.407238,0.007532,0.526821
3,2191,Complete Python 3 Programming Bootcamp: Beginn...,All Levels,4.5,503.0,5106.0,0.871376,0.176791,0.239102,0.007030,0.519569
4,0290,Python Programming - From Basics to Advanced l...,All Levels,4.3,6229.0,319289.0,0.782081,0.152240,0.483884,0.006528,0.518918


# Hybrid Recommender Evaluation

In [82]:
import os 
import pickle
import warnings

import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [ ]:
# Number of Recommendations

K_VALUES = [5, 10]

# Hybrid weights
SEMANTIC_WEIGHT = 0.50
TFIDF_WEIGHT = 0.20
POPULARITY_WEIGHT = 0.20
RATING_WEIGHT = 0.10

# Number of candidates retrieved before ranking
CANDIDATE_K = 100

print("Hybrid configuration:")
print(f"Semantic Weight   : {SEMANTIC_WEIGHT}")
print(f"TF-IDF Weight     : {TFIDF_WEIGHT}")
print(f"Popularity Weight  : {POPULARITY_WEIGHT}")
print(f"Rating Weight     : {RATING_WEIGHT}")
print(f"Candidate Pool    : {CANDIDATE_K}")

Hybrid configuration:
Semantic Weight   : (0.5,)
TF-IDF Weight     : (0.2,)
Popularity Weight  : (0.2,)
Rating Weight     : 0.1
Candidate Pool    : 100


In [84]:
df = df.copy()

# Remove duplicate course IDs
df = df.drop_duplicates(subset="course_id").reset_index(drop=True)

# Make sure course_id is string
df["course_id"] = df["course_id"].astype(str)

In [85]:
TFIDF_VECTORIZER_PATH = "/kaggle/working/tfidf/tfidf_vectorizer.pkl"
TFIDF_MATRIX_PATH =  "/kaggle/working/tfidf/tfidf_matrix.npz"
COURSE_INDEX_PATH = "/kaggle/working/tfidf/course_id_to_index.pkl"


# with open(TFIDF_VECTORIZER_PATH, "rb") as file:
#     tfidf_vectorizer = pickle.load(file)

import joblib
tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)

tfidf_matrix = sparse.load_npz(TFIDF_MATRIX_PATH)

with open(COURSE_INDEX_PATH, "rb") as file:
    course_id_to_index = pickle.load(file)

print("TF-IDF artifacts loaded.")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF artifacts loaded.
TF-IDF matrix shape: (5000, 16505)


In [86]:
EMBEDDINGS_PATH = "/kaggle/working/embeddings/course_embeddings.npy"
course_embeddings = np.load(EMBEDDINGS_PATH)

print("Embeddig shape:", course_embeddings.shape)

Embeddig shape: (5000, 384)


In [87]:
FAISS_PATH =  "/kaggle/working/embeddings/course_index.faiss"

faiss_index = faiss.read_index(FAISS_PATH)
print("FAISS index loaded.")
print("Number of vectors:", faiss_index.ntotal)

FAISS index loaded.
Number of vectors: 5000


In [88]:
course_lookup =  df.set_index("course_id")

print("Number of course",len(course_lookup))

Number of course 5000


In [89]:
def min_max_normalize(series):
    series = series.astype(float)

    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (series - min_value) / (max_value - min_value)

In [ ]:
if "popularity_score" not in df.columns:
    if "students_count" in df.columns:
        df["popularity_score"] = min_max_normalize(
            np.log1p(df["students_count"])
        )
    elif "num_subscribers" in df.columns:
        df["popularity_score"] = min_max_normalize(
            np.log1p(df["num_subscribers"])
        )
    else:
        df["popularity_score"] = 0.0

print(df[["course_id", "popularity_score", "rating_score"]])

     course_id  popularity_score  rating_score
0         0001          0.651820      0.007281
1         0002          0.608427      0.007532
2         0003          0.605177      0.007532
3         0004          0.588268      0.007281
4         0005          0.594198      0.007030
...        ...               ...           ...
4995      4996          0.093642      0.004770
4996      4997          0.138262      0.007030
4997      4998          0.308550      0.005273
4998      4999          0.203876      0.004770
4999      5000          0.221160      0.008285

[5000 rows x 3 columns]


In [91]:
course_id_to_row = {
    course_id: index
    for index, course_id in enumerate(df["course_id"])
}

row_to_course_id = {
    index: course_id
    for course_id, index in course_id_to_row.items()
}

print("Course mappings created.")

Course mappings created.


# Semantic Similarity

In [92]:
def get_semantic_candidates(course_id, candidate_k = CANDIDATE_K):
    if course_id not in course_id_to_row:
        raise ValueError(f"Course id {course_id} not found")

    row_index = course_id_to_row[course_id]

    query_vector = course_embeddings[row_index].astype("float32")

    # faiss search
    distances, indices  = faiss_index.search(
        query_vector.reshape(1, -1),
        candidate_k + 1
    )

    candidates = []

    for distances, index in zip(distances[0], indices[0]):
        if index == -1:
            continue
        candidate_id = row_to_course_id.get(index)

        if candidate_id is None:
            continue
        if candidate_id == course_id:
            continue

        candidates.append({
            "course_id": candidate_id,
            "semantic_score":float(distances)
        })

    return pd.DataFrame(candidates)

# TF-IDF Similarity

In [93]:
def get_tfidf_candidates(course_id, candidate_k=CANDIDATE_K):

    if course_id not in course_id_to_row:
        raise ValueError(f"Course ID {course_id} not found.")

    row_index = course_id_to_row[course_id]

    query_vector = tfidf_matrix[row_index]

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get highest scores
    candidate_indices = np.argsort(
        similarities
    )[::-1][:candidate_k + 1]

    candidates = []

    for index in candidate_indices:

        candidate_id = row_to_course_id.get(index)

        if candidate_id is None:
            continue

        if candidate_id == course_id:
            continue

        candidates.append(
            {
                "course_id": candidate_id,
                "tfidf_score": float(similarities[index])
            }
        )

    return pd.DataFrame(candidates)

# Build hybrid recommendation functions

In [ ]:
def minmax_array(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return values

    min_value = values.min()
    max_value = values.max()

    if max_value == min_value:
        return np.ones_like(values)

    return (values - min_value) / (max_value - min_value)

In [95]:
def hybrid_recommend(
    course_id,
    top_n=10,
    candidate_k=CANDIDATE_K
):

    if course_id not in course_id_to_row:
        raise ValueError(
            f"Course ID {course_id} not found."
        )

    # -----------------------------
    # Semantic candidates
    # -----------------------------

    semantic_df = get_semantic_candidates(
        course_id,
        candidate_k
    )

    # -----------------------------
    # TF-IDF candidates
    # -----------------------------

    tfidf_df = get_tfidf_candidates(
        course_id,
        candidate_k
    )

    # -----------------------------
    # Merge candidates
    # -----------------------------

    candidates = pd.merge(
        semantic_df,
        tfidf_df,
        on="course_id",
        how="outer"
    )

    candidates["semantic_score"] = (
        candidates["semantic_score"]
        .fillna(0)
    )

    candidates["tfidf_score"] = (
        candidates["tfidf_score"]
        .fillna(0)
    )

    # Normalize similarity scores
    candidates["semantic_score"] = minmax_array(
        candidates["semantic_score"]
    )

    candidates["tfidf_score"] = minmax_array(
        candidates["tfidf_score"]
    )

    # -----------------------------
    # Add popularity + rating
    # -----------------------------

    candidates = candidates.merge(
        df[
            [
                "course_id",
                "popularity_score",
                "rating_score"
            ]
        ],
        on="course_id",
        how="left"
    )

    candidates["popularity_score"] = (
        candidates["popularity_score"]
        .fillna(0)
    )

    candidates["rating_score"] = (
        candidates["rating_score"]
        .fillna(0)
    )

    # -----------------------------
    # Hybrid score
    # -----------------------------

    candidates["hybrid_score"] = (

        SEMANTIC_WEIGHT *
        candidates["semantic_score"]

        +

        TFIDF_WEIGHT *
        candidates["tfidf_score"]

        +

        POPULARITY_WEIGHT *
        candidates["popularity_score"]

        +

        RATING_WEIGHT *
        candidates["rating_score"]
    )

    # Sort
    recommendations = (
        candidates
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    # Add course information
    information_columns = [
        "course_id",
        "course_name",
        "level"
    ]

    available_columns = [
        column
        for column in information_columns
        if column in df.columns
    ]

    recommendations = recommendations.merge(
        df[available_columns],
        on="course_id",
        how="left"
    )

    return recommendations

In [96]:
test_course_id = df["course_id"].iloc[0]

recommendations = hybrid_recommend(
    test_course_id,
    top_n = 10
)
display(recommendations)

,course_id,semantic_score,tfidf_score,popularity_score,rating_score,hybrid_score,course_name,level
0,0011,-0.0,-0.162958,0.571622,0.007281,0.082461,The Data Science Course 2022: Complete Data Sc...,All Levels
1,0100,-0.0,-0.160476,0.456252,0.007281,0.059883,REST APIs with Flask and Python in 2022,Intermediate
2,0362,-0.0,-0.158120,0.383904,0.007030,0.045860,The Complete Data Structures and Algorithms Co...,All Levels
3,0305,-0.0,-0.170877,0.386120,0.007532,0.043802,Web Developer Bootcamp with Flask and Python i...,Intermediate
4,0095,-0.0,-0.248358,0.459322,0.007030,0.042896,Spark and Python for Big Data with PySpark,Intermediate
5,0271,-0.0,-0.162336,0.366168,0.006779,0.041444,The Complete Python Masterclass: Learn Python ...,All Levels
6,0415,-0.0,-0.150137,0.352446,0.006779,0.041140,Snowflake[A-Z] Zero to Hero Masterclass(Core+S...,All Levels
7,0478,-0.0,-0.159920,0.356974,0.007281,0.040139,Complete SQL and Databases Bootcamp: Zero to M...,All Levels
8,0407,-0.0,-0.168883,0.364164,0.007030,0.039759,Automated Software Testing with Python,Intermediate
9,0400,-0.0,-0.192153,0.379279,0.007532,0.038178,The Complete Java Development Bootcamp,All Levels


In [97]:
course_name = df.loc[
    df["course_id"] == test_course_id,
    "course_name"
].iloc[0]

print("Selected Course:")
print(course_name)

print("\nRecommended Courses:")

display(
    recommendations[
        [
            "course_id",
            "course_name",
            "hybrid_score"
        ]
    ]
)

Selected Course:
2022 Complete Python Bootcamp From Zero to Hero in Python

Recommended Courses:


,course_id,course_name,hybrid_score
0,0011,The Data Science Course 2022: Complete Data Sc...,0.082461
1,0100,REST APIs with Flask and Python in 2022,0.059883
2,0362,The Complete Data Structures and Algorithms Co...,0.045860
3,0305,Web Developer Bootcamp with Flask and Python i...,0.043802
4,0095,Spark and Python for Big Data with PySpark,0.042896
5,0271,The Complete Python Masterclass: Learn Python ...,0.041444
6,0415,Snowflake[A-Z] Zero to Hero Masterclass(Core+S...,0.041140
7,0478,Complete SQL and Databases Bootcamp: Zero to M...,0.040139
8,0407,Automated Software Testing with Python,0.039759
9,0400,The Complete Java Development Bootcamp,0.038178


# Create Proxy Relevance

In [98]:
def get_relevance_score(
    source_course_id,
    recommended_course_id
):

    source_index = course_id_to_row[source_course_id]
    recommended_index = course_id_to_row[
        recommended_course_id
    ]

    source_vector = tfidf_matrix[source_index]
    recommended_vector = tfidf_matrix[recommended_index]

    similarity = cosine_similarity(
        source_vector,
        recommended_vector
    )[0][0]

    return float(similarity)

In [99]:
RELEVANCE_THRESHOLD = 0.30

In [100]:
def get_relevant_courses(
    source_course_id,
    threshold=RELEVANCE_THRESHOLD
):

    source_index = course_id_to_row[source_course_id]

    source_vector = tfidf_matrix[source_index]

    similarities = cosine_similarity(
        source_vector,
        tfidf_matrix
    ).flatten()

    relevant_indices = np.where(
        similarities >= threshold
    )[0]

    relevant_courses = []

    for index in relevant_indices:

        course_id = row_to_course_id.get(index)

        if course_id is None:
            continue

        if course_id == source_course_id:
            continue

        relevant_courses.append(course_id)

    return set(relevant_courses)

In [101]:
def precision_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    recommended_ids = recommended_ids[:k]

    if len(recommended_ids) == 0:
        return 0.0

    relevant_count = sum(
        course_id in relevant_ids
        for course_id in recommended_ids
    )

    return relevant_count / len(recommended_ids)

In [102]:
def recall_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    if len(relevant_ids) == 0:
        return 0.0

    recommended_ids = recommended_ids[:k]

    relevant_count = sum(
        course_id in relevant_ids
        for course_id in recommended_ids
    )

    return relevant_count / len(relevant_ids)

In [103]:
def f1_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    precision = precision_at_k(
        recommended_ids,
        relevant_ids,
        k
    )

    recall = recall_at_k(
        recommended_ids,
        relevant_ids,
        k
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [104]:
def hit_rate_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    recommended_ids = recommended_ids[:k]

    for course_id in recommended_ids:

        if course_id in relevant_ids:
            return 1.0

    return 0.0

In [105]:
def dcg_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    recommended_ids = recommended_ids[:k]

    dcg = 0.0

    for rank, course_id in enumerate(
        recommended_ids,
        start=1
    ):

        if course_id in relevant_ids:

            dcg += 1 / np.log2(rank + 1)

    return dcg

In [106]:
def ndcg_at_k(
    recommended_ids,
    relevant_ids,
    k
):

    if len(relevant_ids) == 0:
        return 0.0

    actual_dcg = dcg_at_k(
        recommended_ids,
        relevant_ids,
        k
    )

    ideal_recommendations = list(
        relevant_ids
    )[:k]

    ideal_dcg = dcg_at_k(
        ideal_recommendations,
        relevant_ids,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [107]:
def evaluate_course(
    course_id,
    k=10
):

    recommendations = hybrid_recommend(
        course_id,
        top_n=k
    )

    recommended_ids = (
        recommendations["course_id"]
        .astype(str)
        .tolist()
    )

    relevant_ids = get_relevant_courses(
        course_id
    )

    return {
        "course_id": course_id,

        "precision": precision_at_k(
            recommended_ids,
            relevant_ids,
            k
        ),

        "recall": recall_at_k(
            recommended_ids,
            relevant_ids,
            k
        ),

        "f1": f1_at_k(
            recommended_ids,
            relevant_ids,
            k
        ),

        "hit_rate": hit_rate_at_k(
            recommended_ids,
            relevant_ids,
            k
        ),

        "ndcg": ndcg_at_k(
            recommended_ids,
            relevant_ids,
            k
        )
    }

In [108]:
N_EVALUATION_COURSES = 100

np.random.seed(42)

evaluation_courses = np.random.choice(
    df["course_id"].values,
    size=min(
        N_EVALUATION_COURSES,
        len(df)
    ),
    replace=False
)

print(
    "Courses selected for evaluation:",
    len(evaluation_courses)
)

Courses selected for evaluation: 100


In [ ]:
evaluation_results = []
evaluation_errors = []

for i, course_id in enumerate(
    evaluation_courses
):

    try:

        result = evaluate_course(
            course_id,
            k=10
        )

        evaluation_results.append(result)

    except Exception as error:

        evaluation_errors.append({
            "course_id": course_id,
            "error": str(error)
        })

    if (i + 1) % 20 == 0:

        print(
            f"Processed {i + 1}/"
            f"{len(evaluation_courses)}"
        )

print(f"Evaluation errors: {len(evaluation_errors)}")
if evaluation_errors:
    display(pd.DataFrame(evaluation_errors).head())

Processed 20/100
Processed 40/100
Processed 60/100
Processed 80/100
Processed 100/100


In [110]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

display(evaluation_df.head())

,course_id,precision,recall,f1,hit_rate,ndcg
0,1502,0.0,0.0,0.0,0.0,0.0
1,2587,0.0,0.0,0.0,0.0,0.0
2,2654,0.0,0.0,0.0,0.0,0.0
3,1056,0.0,0.0,0.0,0.0,0.0
4,0706,0.0,0.0,0.0,0.0,0.0


In [111]:
final_metrics = {

    "Precision@10":
        evaluation_df["precision"].mean(),

    "Recall@10":
        evaluation_df["recall"].mean(),

    "F1@10":
        evaluation_df["f1"].mean(),

    "Hit Rate@10":
        evaluation_df["hit_rate"].mean(),

    "NDCG@10":
        evaluation_df["ndcg"].mean()
}

metrics_df = pd.DataFrame(
    final_metrics.items(),
    columns=[
        "Metric",
        "Score"
    ]
)

metrics_df["Score"] = metrics_df[
    "Score"
].round(4)

display(metrics_df)

,Metric,Score
0,Precision@10,0.0
1,Recall@10,0.0
2,F1@10,0.0
3,Hit Rate@10,0.0
4,NDCG@10,0.0


In [ ]:
all_k_results = []
all_k_errors = []

for k in [5, 10]:

    for course_id in evaluation_courses:

        try:

            recommendations = hybrid_recommend(
                course_id,
                top_n=k
            )

            recommended_ids = (
                recommendations["course_id"]
                .astype(str)
                .tolist()
            )

            relevant_ids = get_relevant_courses(
                course_id
            )

            all_k_results.append({

                "course_id": course_id,
                "k": k,

                "precision": precision_at_k(
                    recommended_ids,
                    relevant_ids,
                    k
                ),

                "recall": recall_at_k(
                    recommended_ids,
                    relevant_ids,
                    k
                ),

                "f1": f1_at_k(
                    recommended_ids,
                    relevant_ids,
                    k
                ),

                "hit_rate": hit_rate_at_k(
                    recommended_ids,
                    relevant_ids,
                    k
                ),

                "ndcg": ndcg_at_k(
                    recommended_ids,
                    relevant_ids,
                    k
                )
            })

        except Exception as error:
            all_k_errors.append({
                "course_id": course_id,
                "k": k,
                "error": str(error)
            })


all_k_results_df = pd.DataFrame(
    all_k_results
)

print(f"Evaluation errors across k values: {len(all_k_errors)}")
if all_k_errors:
    display(pd.DataFrame(all_k_errors).head())

display(all_k_results_df.head())

,course_id,k,precision,recall,f1,hit_rate,ndcg
0,1502,5,0.0,0.0,0.0,0.0,0.0
1,2587,5,0.0,0.0,0.0,0.0,0.0
2,2654,5,0.0,0.0,0.0,0.0,0.0
3,1056,5,0.0,0.0,0.0,0.0,0.0
4,0706,5,0.0,0.0,0.0,0.0,0.0


In [113]:
summary_metrics = (
    all_k_results_df
    .groupby("k")
    [
        [
            "precision",
            "recall",
            "f1",
            "hit_rate",
            "ndcg"
        ]
    ]
    .mean()
    .reset_index()
)

summary_metrics = summary_metrics.round(4)

display(summary_metrics)

,k,precision,recall,f1,hit_rate,ndcg
0,5,0.0,0.0,0.0,0.0,0.0
1,10,0.0,0.0,0.0,0.0,0.0


# Recommendation Coverage¶
How much of the entire course catalog does my recommender actually recommend?

In [114]:
def calculate_coverage(
    evaluation_courses,
    top_n=10
):

    recommended_courses = set()

    for course_id in evaluation_courses:

        try:

            recommendations = hybrid_recommend(
                course_id,
                top_n=top_n
            )

            recommended_courses.update(
                recommendations["course_id"]
                .astype(str)
                .tolist()
            )

        except Exception:
            continue

    total_courses = df["course_id"].nunique()

    coverage = (
        len(recommended_courses)
        / total_courses
    )

    return coverage

In [115]:
coverage_at_10 = calculate_coverage(
    evaluation_courses,
    top_n=10
)

print(
    f"Catalog Coverage@10: "
    f"{coverage_at_10:.4f}"
)

print(
    f"Percentage: "
    f"{coverage_at_10 * 100:.2f}%"
)

Catalog Coverage@10: 0.1392
Percentage: 13.92%


Diversity@K

We don't want the recommender to return 10 almost-identical courses.

We'll calculate Intra-List Diversity using TF-IDF.

In [116]:
def intra_list_diversity(
    recommended_ids
):

    if len(recommended_ids) <= 1:
        return 0.0

    indices = [
        course_id_to_row[course_id]
        for course_id in recommended_ids
        if course_id in course_id_to_row
    ]

    if len(indices) <= 1:
        return 0.0

    vectors = tfidf_matrix[indices]

    similarity_matrix = cosine_similarity(
        vectors
    )

    # Get upper triangle excluding diagonal
    upper_triangle = similarity_matrix[
        np.triu_indices_from(
            similarity_matrix,
            k=1
        )
    ]

    if len(upper_triangle) == 0:
        return 0.0

    # Diversity = 1 - average similarity
    diversity = 1 - upper_triangle.mean()

    return float(diversity)

Calculate Average Diversity

In [117]:
diversity_scores = []

for course_id in evaluation_courses:

    try:

        recommendations = hybrid_recommend(
            course_id,
            top_n=10
        )

        recommended_ids = (
            recommendations["course_id"]
            .astype(str)
            .tolist()
        )

        diversity = intra_list_diversity(
            recommended_ids
        )

        diversity_scores.append(diversity)

    except Exception:
        continue


average_diversity = np.mean(
    diversity_scores
)

print(
    f"Average Intra-List Diversity@10: "
    f"{average_diversity:.4f}"
)


Average Intra-List Diversity@10: 0.9384


Average Intra-List Diversity@10: 0.9384

Final Evaluation Dashboard

In [118]:
final_evaluation = pd.DataFrame({

    "Metric": [
        "Precision@5",
        "Recall@5",
        "F1@5",
        "Hit Rate@5",
        "NDCG@5",

        "Precision@10",
        "Recall@10",
        "F1@10",
        "Hit Rate@10",
        "NDCG@10",

        "Coverage@10",
        "Diversity@10"
    ],

    "Score": [

        summary_metrics.loc[
            summary_metrics["k"] == 5,
            "precision"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 5,
            "recall"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 5,
            "f1"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 5,
            "hit_rate"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 5,
            "ndcg"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 10,
            "precision"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 10,
            "recall"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 10,
            "f1"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 10,
            "hit_rate"
        ].iloc[0],

        summary_metrics.loc[
            summary_metrics["k"] == 10,
            "ndcg"
        ].iloc[0],

        coverage_at_10,

        average_diversity
    ]
})

final_evaluation["Score"] = (
    final_evaluation["Score"]
    .round(4)
)

display(final_evaluation)

,Metric,Score
0,Precision@5,0.0000
1,Recall@5,0.0000
2,F1@5,0.0000
3,Hit Rate@5,0.0000
4,NDCG@5,0.0000
5,Precision@10,0.0000
6,Recall@10,0.0000
7,F1@10,0.0000
8,Hit Rate@10,0.0000
9,NDCG@10,0.0000


Save Evaluation Results

In [ ]:
OUTPUT_PATH = "/kaggle/working/evaluation_path/evaluation"

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

final_evaluation.to_csv(
    f"{OUTPUT_PATH}/hybrid_evaluation.csv",
    index=False
)

summary_metrics.to_csv(
    f"{OUTPUT_PATH}/hybrid_metrics_at_k.csv",
    index=False
)

print("Evaluation results saved successfully.")

Evaluation results saved successfully.


test a Real Course

Finally, show the recommender like your Streamlit application will use it.

In [120]:
sample_course_id = df["course_id"].iloc[10]

sample_course = df[
    df["course_id"] == sample_course_id
].iloc[0]

print("=" * 70)
print("SELECTED COURSE")
print("=" * 70)

print(
    f"Course ID   : {sample_course['course_id']}"
)

print(
    f"Course Name : {sample_course['course_name']}"
)

print("\n")
print("=" * 70)
print("HYBRID RECOMMENDATIONS")
print("=" * 70)

result = hybrid_recommend(
    sample_course_id,
    top_n=10
)

display(
    result[
        [
            "course_id",
            "course_name",
            "semantic_score",
            "tfidf_score",
            "popularity_score",
            "rating_score",
            "hybrid_score"
        ]
    ]
)

SELECTED COURSE
Course ID   : 0011
Course Name : The Data Science Course 2022: Complete Data Science Bootcamp


HYBRID RECOMMENDATIONS


,course_id,course_name,semantic_score,tfidf_score,popularity_score,rating_score,hybrid_score
0,0051,Data Science A-Z™: Real-Life Data Science Exer...,-0.0,-0.330977,0.500130,0.007281,0.034559
1,0039,Deep Learning A-Z™: Hands-On Artificial Neural...,-0.0,-0.373103,0.528539,0.007030,0.031790
2,0227,R Programming: Advanced Analytics In R For Dat...,-0.0,-0.286643,0.411117,0.007281,0.025623
3,0154,Introduction to Machine Learning for Data Science,-0.0,-0.295955,0.419510,0.006779,0.025389
4,1048,Python In Practice : 15 Projects to Master Python,-0.0,-0.290244,0.392780,0.007532,0.021260
5,1283,Python & Introduction to Data Science,-0.0,-0.285829,0.356812,0.007030,0.014900
6,0670,Python for Statistical Analysis,-0.0,-0.321453,0.381260,0.006779,0.012639
7,0188,Statistics for Business Analytics and Data Sci...,-0.0,-0.356502,0.415277,0.007030,0.012458
8,0315,Probability and Statistics for Business and Da...,-0.0,-0.329184,0.372241,0.007281,0.009340
9,0684,Data Science Career Guide - Interview Preparation,-0.0,-0.291979,0.334808,0.006779,0.009244
